### CreateSources (post-cutover, oxjob #548)

The sources registry lives in the **openalex-sources** Heroku Postgres and is maintained by
that app's feed / merge / curation jobs. This notebook no longer *builds* sources — it
materializes a daily, run-consistent snapshot of the federated registry
(`openalex_sources.public.sources`) in the legacy 40-column shape, so downstream consumers
(CreateSourcesApi, CreateLocationsWithSources, CreateWorksBase, CreateInstitutionsApi) run
unchanged. Runs as a plain SQL-warehouse task (`source: GIT`) — the DLT pipeline is retired.

Legacy-shape contract: `issn` = registry `issn_l`; `webpage` = `homepage_url`;
`issns` = registry array, with ISSN-less rows NULL except the legacy-[] cohort (`legacy_empty_issns_sources`) — per-row parity with pre-cutover works embeds; JSONB columns parsed to typed arrays; date columns
cast to string. Merged sources are **included** as redirect rows (`merge_into_id` set);
consumers needing active-only filter `merge_into_id IS NULL`.


In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.sources.sources')
AS SELECT
  s.id,
  endpoint_id,
  display_name,
  issn_l AS issn,
  publisher,
  homepage_url AS webpage,
  is_oa,
  type,
  from_json(apc_prices, 'array<struct<price:int,currency:string>>') AS apc_prices,
  is_society_journal,
  from_json(societies, 'array<struct<url:string,organization:string>>') AS societies,
  apc_usd,
  fatcat_id,
  wikidata_id,
  crossref_id,
  country,
  country_code,
  from_json(alternate_titles, 'array<string>') AS alternate_titles,
  publisher_id,
  institution_id,
  is_core,
  merge_into_id,
  merge_into_date,
  CAST(updated_date AS string) AS updated_date,
  CAST(created_date AS string) AS created_date,
  display_name_before_override,
  override_timestamp,
  datacite_id,
  -- legacy per-row representation (oxjob #548 churn fix): ISSN-less sources
  -- embed NULL in works, except the 2,912 pre-cutover sources that embedded []
  -- (see openalex.sources.legacy_empty_issns_sources) -- retire deliberately
  COALESCE(s.issns, CASE WHEN leg.id IS NOT NULL THEN array() END) AS issns,
  is_in_doaj,
  is_in_doaj_start_year,
  doaj_license,
  is_in_scielo,
  is_ojs,
  is_oa_high_oa_rate,
  high_oa_rate_start_year,
  is_fully_open_in_jstage,
  sample_pmh_record,
  COALESCE(from_json(datacite_ids, 'array<string>'), array()) AS datacite_ids,
  is_preprint_repository
FROM openalex_sources.public.sources s
LEFT JOIN openalex.sources.legacy_empty_issns_sources leg ON leg.id = s.id

In [ ]:
SELECT
  COUNT(*) AS total,
  COUNT_IF(merge_into_id IS NULL) AS active,
  COUNT_IF(is_in_doaj) AS in_doaj,
  MAX(id) AS max_id
FROM identifier('openalex' || :env_suffix || '.sources.sources')